# Tome 5, Chapter 11: Exponential sums (Python companion)

Exercises first, solutions at the end. We walk the curlicues $\sum e(\alpha n^2)$ with exact phases, measure how evenly $n^2\sqrt2$, $n^{3/2}$, $\sqrt n$ and $\log n$ fill the circle (Theorems 11.4 and 11.10, Exercise 11.6), compare Weyl's inequality (Theorem 11.7) and van der Corput's test (Theorem 11.9) with the truth, test the Pólya–Vinogradov inequality (Theorem 11.11, Exercise 11.4), hunt large least non-residues (Theorem 11.12), and finally take square roots modulo $63$-bit primes, where the least non-residue is a running time. Requirements: numpy, sympy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P11.1 (★) Walks.** Compute $|\sum_{n\le N}e(\alpha n^2)|$ for $N=10^3,10^4,10^5$ and $\alpha=\sqrt2,\frac{\sqrt2}2,\frac1\pi,\frac{\sqrt5-1}2$. Floating point is not enough for $\alpha n^2$ modulo $1$ when $n^2$ is large: use an exact integer approximation of $\alpha$.

**P11.2 (★) Discrepancy.** For $u_n$ among $n^2\sqrt2$, $n\sqrt2$, $n^{3/2}$, $\sqrt n$, $\frac{n^2}5$, $\log n$, compute the discrepancy $\sup_t\bigl|\frac1N\#\{n\le N:\{u_n\}<t\}-t\bigr|$ for $N=10^5$. Which sequences are equidistributed, and which converge fastest?

**P11.3 (★★) Weyl's inequality in practice.** Compute the convergents of $\sqrt2$ and $\frac1\pi$, and compare $|S|$ with the bound of Theorem 11.7, using the largest convergent denominator $q\le N$. Why is the bound so poor for $\frac1\pi$ at $N=10^5$?

**P11.4 (★) Van der Corput.** Compare $|\sum e(n^{3/2})|$ over $n\le N$ and over $[N,2N]$ with Theorem 11.9. Check Lemma 11.8 on $f(x)=\frac{x\log x}{10}$ over $[1000,2000]$.

**P11.5 (★★) Character sums.** For primes $p$ up to $10^7$, compute the largest $|\sum_{n\in I}(\frac np)|$ over intervals $I$, and compare with $\sqrt p$ and $\sqrt p(1+\log p)$. Check the identity of Exercise 11.4.

**P11.6 (★) Least non-residues.** Find the records of $n(p)$ for $p<10^6$, and the average of $n(p)$. Compare with Theorem 11.12, with Bach's $2\log^2p$, and with Erdős's constant $\sum_k\frac{p_k}{2^k}$.

**P11.7 (★★) CS application: square roots modulo $p$.** The Tonelli–Shanks algorithm computes square roots modulo a prime, and needs one quadratic non-residue, found by trying $2,3,4,\dots$ Implement it, run it on $2000$ random $63$-bit primes, and record how many candidates it tries.

## Solutions

### P11.1

In [ ]:
import math, cmath, time
import numpy as np
import mpmath as mp
mp.mp.dps = 50

def weyl_partial_sums(alpha, N, bits=80):
    # exact phases: alpha*n^2 modulo 1 computed with a 80-bit integer approximation of alpha
    A = int(mp.floor(alpha * 2**bits)); M = 2**bits
    out = np.empty(N, complex); s = 0j
    for n in range(1, N + 1):
        s += cmath.exp(2j * math.pi * ((A * n * n) % M) / M)
        out[n - 1] = s
    return out

alphas = {"sqrt2": mp.sqrt(2), "sqrt2/2": mp.sqrt(2) / 2, "1/pi": 1 / mp.pi, "golden": (mp.sqrt(5) - 1) / 2}
sums = {k: weyl_partial_sums(a, 10**5) for k, a in alphas.items()}
print("            N:      1e3      1e4      1e5   |S|/sqrt(N) at 1e5")
for k, S in sums.items():
    print(f"{k:>12s}: " + " ".join(f"{abs(S[N - 1]):8.1f}" for N in (10**3, 10**4, 10**5)) + f"   {abs(S[-1]) / math.sqrt(10**5):6.3f}")

The golden ratio gives the smallest sums, and $\frac1\pi$ the largest for small $N$: its convergent $\frac{113}{355}$ is extraordinarily good (the next partial quotient is $292$), so for $N$ up to about $10^5$ the walk behaves as if $\alpha$ were $\frac{113}{355}$, a chain of spirals with period $355$. All values are of the order of $\sqrt N$, as for a random walk: at $N=10^5$, $|S|/\sqrt N$ lies between $0.2$ and $1.4$. Floating point would fail here: at $n=10^5$, $n^2\alpha$ is about $10^{10}$, and a double keeps only $6$ digits after the point.

### P11.2

In [ ]:
def discrepancy(u):
    # max over [0,t) of |proportion - t|, for the fractional parts u
    v = np.sort(np.mod(u, 1.0)); n = len(v); i = np.arange(1, n + 1)
    return max(np.max(i / n - v), np.max(v - (i - 1) / n))
N = 10**5
n = np.arange(1, N + 1, dtype=np.float64)
A = int(mp.floor(mp.sqrt(2) * 2**80))
sq2 = np.array([((A * k * k) % 2**80) / 2**80 for k in range(1, N + 1)])
tests = {"n^2 sqrt2": sq2, "n sqrt2": np.mod(n * math.sqrt(2), 1), "n^(3/2)": np.mod(n**1.5, 1),
         "sqrt(n)": np.mod(np.sqrt(n), 1), "n^2/5": np.mod(n * n / 5, 1), "log n": np.mod(np.log(n), 1)}
for name, u in tests.items():
    print(f"{name:>10s}: discrepancy over n <= 1e5 = {discrepancy(u):.4f}")

Five behaviours. $n\sqrt2$ is extremely regular (discrepancy $O(\frac{\log N}N)$, by the three-gap structure of Tome 4); $n^2\sqrt2$ and $\sqrt n$ are equidistributed with discrepancy of order $N^{-1/2}$; $n^{3/2}$ converges more slowly, as the bound $N^{c/2}=N^{3/4}$ of Theorem 11.10 suggests; $\frac{n^2}5$ is stuck on three values (Exercise 11.2); and $\log n$ is not equidistributed at all (Exercise 11.6): its discrepancy oscillates and never tends to $0$.

### P11.3

In [ ]:
def convergents(x, k):
    a = []; y = x
    for _ in range(k):
        q = int(mp.floor(y)); a.append(q); y = 1 / (y - q)
    p0, q0, p1, q1 = 1, 0, a[0], 1; out = [(p1, q1)]
    for t in a[1:]:
        p0, q0, p1, q1 = p1, q1, t * p1 + p0, t * q1 + q0; out.append((p1, q1))
    return a, out
for name in ("sqrt2", "1/pi"):
    a, cv = convergents(alphas[name], 8)
    print(name, "partial quotients", a, "\n   convergents", cv)
def weyl_bound(N, q):
    return math.sqrt(N + 2 * (2 * N / q + 1) * (4 * N + 2 * q + 2 * q * math.log(q)))
for name in ("sqrt2", "1/pi"):
    a, cv = convergents(alphas[name], 12)
    for N in (10**3, 10**4, 10**5):
        p, q = max((pq for pq in cv if pq[1] <= N), key=lambda t: t[1])
        print(f"{name:>6s} N={N:6d}: |S| = {abs(sums[name][N - 1]):7.1f}, q = {q:6d}, Weyl bound {weyl_bound(N, q):8.1f}, sqrt(N log N) = {math.sqrt(N * math.log(N)):6.1f}")

For $\sqrt2$, Weyl's bound is about four times $\sqrt{N\log N}$ and ten to thirty-five times the truth: the constants in Lemma 11.6 are generous, but the order $\sqrt{N\log N}$ is right (Exercise 11.3). For $\frac1\pi$ and $N=10^5$, the best denominator below $N$ is $q=355$, and the next convergent has $q=103\,993>N$. The theorem then only gives $\frac N{\sqrt q}$, as it must: up to $N\approx10^5$ the sum truly behaves like a sum with rational $\alpha=\frac{113}{355}$.

### P11.4

In [ ]:
def S_smooth(f, N):
    n = np.arange(1, N + 1, dtype=np.float64)
    return abs(np.exp(2j * np.pi * np.mod(f(n), 1)).sum())
for N in (10**3, 10**4, 10**5, 10**6):
    lam = 0.75 / math.sqrt(2 * N)          # f(x) = x^(3/2): f'' = (3/4) x^(-1/2) on [N, 2N]
    blk = abs(np.exp(2j * np.pi * np.mod(np.arange(N, 2 * N + 1, dtype=np.float64)**1.5, 1)).sum())
    print(f"N={N:8d}: |sum_(n<=N) e(n^1.5)| = {S_smooth(lambda x: x**1.5, N):8.1f}   block [N,2N]: {blk:8.1f}, "
          f"second derivative bound {4 * math.sqrt(2) * N * math.sqrt(lam) + 12 / math.sqrt(lam):8.1f}")
print("Kusmin-Landau: f(x) = x log x / 10 on [1000, 2000]: f' in", round(math.log(1000) / 10 + 0.1, 3), "to", round(math.log(2000) / 10 + 0.1, 3))
x = np.arange(1000, 2001, dtype=np.float64)
d = min(math.log(1000) / 10 + 0.1, 1 - (math.log(2000) / 10 + 0.1))
print("   |sum| =", round(abs(np.exp(2j * np.pi * np.mod(x * np.log(x) / 10, 1)).sum()), 2), " bound 1 + 1/delta =", round(1 + 1 / d, 2))

The second derivative test is right about the order: on $[N,2N]$, $\lambda\approx N^{-1/2}$, the bound is of order $N\sqrt\lambda=N^{3/4}$, and the true sums grow like $N^{3/4}$ too (a factor $5.7$ for each factor $10$, and $10^{3/4}=5.6$); our constants cost a factor of about $15$. The Kusmin–Landau example has $f'$ between $0.79$ and $0.86$, safely away from the integers, and the sum of $1001$ terms is $1.35$: smaller than the bound $8.15$, and far smaller than $\sqrt{1001}$. When the phase never resonates, there is \emph{more} than square-root cancellation.

### P11.5

In [ ]:
from sympy import primerange
def legendre_table(p):
    t = np.full(p, -1, dtype=np.int64); t[0] = 0
    t[(np.arange(1, p, dtype=np.int64)**2) % p] = 1
    return t
print("        p   max interval sum   / sqrt p   / (sqrt p (1+log p))")
for p in (1009, 10007, 100003, 1000003, 10000019):
    S = np.concatenate(([0], np.cumsum(legendre_table(p)[1:])))
    m = S.max() - S.min()
    print(f"{p:9d} {m:12d} {m / math.sqrt(p):12.3f} {m / (math.sqrt(p) * (1 + math.log(p))):14.4f}")
p = 10007; chi = legendre_table(p); N = p // 2
ext = np.concatenate((chi, chi))
win = np.convolve(ext, np.ones(N, dtype=np.int64), 'valid')[1:p + 1]
print("mean square over all shifts, N = (p-1)/2:", (win**2).sum(), " N(p-N) =", N * (p - N))

The largest interval sum is about $1.2\sqrt p$ to $1.3\sqrt p$ at every height: the factor $\log p$ of Theorem 11.11 is not seen in these examples. (It is needed in general: Paley showed in 1932 that for infinitely many $p$ the maximum exceeds a constant times $\sqrt p\log\log p$, and Montgomery and Vaughan showed that $\sqrt p\log\log p$ is the truth under the generalised Riemann hypothesis.) The mean square identity $N(p-N)$ of Exercise 11.4 holds exactly.

### P11.6

In [ ]:
def least_nonres(p):
    n = 2
    while pow(n, (p - 1) // 2, p) == 1: n += 1
    return n
t = time.time()
P = list(primerange(3, 10**6)); vals = [least_nonres(p) for p in P]
best = 0
for p, v in zip(P, vals):
    if v > best:
        best = v; print(f"record n(p) = {v:3d} at p = {p:7d}   log n(p)/log p = {math.log(v) / math.log(p):.3f}   2 log^2 p = {2 * math.log(p)**2:.0f}")
print(f"mean of n(p) over the {len(P)} odd primes below 1e6: {np.mean(vals):.4f}")
pr = list(primerange(2, 400))
print("Erdős's constant sum_k p_k / 2^k =", float(sum(mp.mpf(q) / 2**(k + 1) for k, q in enumerate(pr))))
print(f"({time.time() - t:.0f} s)")

The records grow very slowly: the exponent $\frac{\log n(p)}{\log p}$ of the records falls from $0.63$ to $0.29$, and the records stay far below Bach's $2\log^2p$. On average $n(p)$ is a small constant: $2$ is a non-residue for half the primes, $3$ for half of the rest, and so on, so the mean should be $\frac22+\frac34+\frac58+\cdots=\sum_k\frac{p_k}{2^k}=3.6746$, if the residue characters of different primes behave independently. Erdős proved this in 1961; the data give $3.646$.

### P11.7

In [ ]:
import random
random.seed(5)
def tonelli_shanks(a, p):
    # square root of a quadratic residue a modulo an odd prime p; returns (root, tries to find a non-residue)
    q, s = p - 1, 0
    while q % 2 == 0: q //= 2; s += 1
    z, tries = 2, 1
    while pow(z, (p - 1) // 2, p) != p - 1: z += 1; tries += 1
    m, c, t, r = s, pow(z, q, p), pow(a, q, p), pow(a, (q + 1) // 2, p)
    while t != 1:
        i, t2 = 0, t
        while t2 != 1: t2 = t2 * t2 % p; i += 1
        b = pow(c, 1 << (m - i - 1), p)
        m, c, t, r = i, b * b % p, t * b * b % p, r * b % p
    return r, tries
from sympy import randprime
tries = []
for _ in range(2000):
    p = randprime(2**62, 2**63)
    a = pow(random.randrange(2, p), 2, p)
    r, k = tonelli_shanks(a, p)
    assert r * r % p == a
    tries.append(k)
print("Tonelli-Shanks on 2000 random 63-bit primes: mean tries to find a non-residue", np.mean(tries), " max", max(tries))
print("distribution of the number of tries:", np.bincount(tries)[1:8])

Half the primes stop at the first try ($2$ is a non-residue exactly when $p\equiv\pm3\pmod8$), a quarter at the second, and the counts keep halving: the tries are almost a geometric law, with the composites $4,8,9,\dots$ never stopping the search since they are squares or products of residues. The mean is about $2.7$ tries; the worst case among $2000$ primes was $40$. Without the generalised Riemann hypothesis, the best proven bound for the search is Burgess's $p^{0.152}$, which for $63$-bit primes is about $750$: Tonelli–Shanks is a randomised algorithm in theory and a deterministic one in practice, and Theorem 11.12 is the first step toward explaining why.